In [1]:
# Cell 1 -- paths + check everything exists
import sys, json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from ultralytics.utils.metrics import box_iou, ap_per_class

sys.path.insert(0, r"C:\Users\LENOVO\Documents\NanoDet")
from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model
from nanodet.evaluator import CocoDetectionEvaluator

PROJ = Path(r"C:\Users\LENOVO\Documents\Thermal_Fusion_Paper")
DATA = Path(r"C:\Users\LENOVO\Documents\Datasets")
CK = PROJ / "checkpoints"
OUT_DIR = PROJ / "journal" / "results"
EVAL_SCRATCH = PROJ / "journal" / "nanodet_eval_scratch"   # pycocotools writes its temp json here
OUT_DIR.mkdir(parents=True, exist_ok=True); EVAL_SCRATCH.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 416

SETUPS = {
    "flir": {
        "model_cfg": PROJ / r"configs\model_only_config.yml",
        "config_a": PROJ / r"results\nanodet_config_a_flir\model_best\model_best.ckpt",
        "rgb_dir": DATA / r"flir_align\visible\test",
        "thermal_dir": DATA / r"flir_align\thermal\test",
        "splits": {"night": DATA / r"flir_align\coco_annotations\test_new_intid_night.json",
                   "full":  DATA / r"flir_align\coco_annotations\test_new_intid.json"},
        "class_names": ["car", "person", "bicycle"],
        "models": {
            ("B", 1): ("B", CK / "config_b_nanodet_single_fusion_best.pt", "thermal"),
            ("B", 2): ("B", CK / "config_b_nanodet_single_fusion_seed2_best.pt", "thermal"),
            ("C", 1): ("C", CK / "config_c_nanodet_multi_fusion_best.pt", "thermal"),
            ("C", 2): ("C", CK / "config_c_nanodet_multi_fusion_seed2_best.pt", "thermal"),
            ("B_rgbgray", 1): ("B", CK / "config_b_nanodet_rgbgray_flir_s1_best.pt", "rgbgray"),
        },
    },
    "m3fd": {
        "model_cfg": PROJ / r"configs\model_only_config_m3fd.yml",
        "config_a": CK / "NanoDet_RGB_M3FD_model_best.ckpt",
        "rgb_dir": DATA / r"M3FD_Detection\Vis",
        "thermal_dir": DATA / r"M3FD_Detection\Ir",
        "splits": {"full": DATA / r"M3FD_Detection\coco_annotations\val_m3fd.json"},
        "class_names": ["Bus", "Car", "Lamp", "Motorcycle", "People", "Truck"],
        "models": {
            ("B", 1): ("B", CK / "config_b_nanodet_single_fusion_m3fd_best.pt", "thermal"),
            ("C", 1): ("C", CK / "config_c_nanodet_multi_fusion_m3fd_best.pt", "thermal"),
            ("B_rgbgray", 1): ("B", CK / "config_b_nanodet_rgbgray_m3fd_s1_best.pt", "rgbgray"),
        },
    },
}

missing = []
for name, s in SETUPS.items():
    paths = [s["model_cfg"], s["config_a"], s["rgb_dir"], s["thermal_dir"], *s["splits"].values(),
             *[m[1] for m in s["models"].values()]]
    for p in paths:
        if not Path(p).exists():
            missing.append(p)
print("all paths found" if not missing else "MISSING:\n" + "\n".join(map(str, missing)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

C:\Users\LENOVO\miniconda3\envs\cv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


all paths found
Using device: cuda


In [2]:
# Cell 2 -- eval code (same metrics as review_part2.0)
IOUV = torch.linspace(0.5, 0.95, 10)
NORM_MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32).reshape(1, 1, 3)
NORM_STD = np.array([57.375, 57.12, 58.395], dtype=np.float32).reshape(1, 1, 3)


def _match_predictions(pred_classes, true_classes, iou, iouv):
    correct = np.zeros((pred_classes.shape[0], iouv.shape[0])).astype(bool)
    iou = (iou * (true_classes[:, None] == pred_classes)).cpu().numpy()
    for i, threshold in enumerate(iouv.cpu().tolist()):
        matches = np.array(np.nonzero(iou >= threshold)).T
        if matches.shape[0]:
            if matches.shape[0] > 1:
                matches = matches[iou[matches[:, 0], matches[:, 1]].argsort()[::-1]]
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
            correct[matches[:, 1].astype(int), i] = True
    return torch.from_numpy(correct)


def nanodet_results_to_pr(all_results, coco_api, cat_ids, class_names):
    """Same as review_part2.0: ultralytics-style P/R/mAP from NanoDet detections."""
    all_tp, all_conf, all_pred_cls, all_target_cls = [], [], [], []
    for image_id, dets in all_results.items():
        anns = coco_api.loadAnns(coco_api.getAnnIds(imgIds=[image_id]))
        gt_boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]
        gt_cls = [a["category_id"] for a in anns]
        gt_boxes_t = torch.tensor(gt_boxes, dtype=torch.float32) if gt_boxes else torch.zeros((0, 4))
        gt_cls_t = torch.tensor(gt_cls, dtype=torch.float32) if gt_cls else torch.zeros((0,))
        all_target_cls.append(gt_cls_t.numpy())
        pb, pc, pcls = [], [], []
        for label, bboxes in dets.items():
            for bbox in bboxes:
                pb.append(bbox[:4]); pc.append(bbox[4]); pcls.append(cat_ids[label])
        if not pb:
            continue
        pb_t = torch.tensor(pb, dtype=torch.float32)
        pcls_t = torch.tensor(pcls, dtype=torch.float32)
        if gt_cls_t.shape[0] == 0:
            tp = np.zeros((pb_t.shape[0], IOUV.shape[0]), dtype=bool)
        else:
            tp = _match_predictions(pcls_t, gt_cls_t, box_iou(gt_boxes_t, pb_t), IOUV).numpy()
        all_tp.append(tp); all_conf.append(np.array(pc, dtype=np.float32)); all_pred_cls.append(pcls_t.numpy())
    names_by_id = {cat_ids[i]: n for i, n in enumerate(class_names)}
    if not all_tp:   # no detections at all
        return {"precision": 0.0, "recall": 0.0, "map50": 0.0, "map50_95": 0.0, "per_class_ap50": {}}
    tp, fp, p, r, f1, ap, uc = ap_per_class(np.concatenate(all_tp), np.concatenate(all_conf),
                                            np.concatenate(all_pred_cls), np.concatenate(all_target_cls),
                                            names=names_by_id)[:7]
    return {"precision": float(p.mean()), "recall": float(r.mean()),
            "map50": float(ap[:, 0].mean()), "map50_95": float(ap.mean()),
            "per_class_ap50": {names_by_id[int(c)]: float(ap[i, 0]) for i, c in enumerate(uc)}}


class PairedEvalDataset(Dataset):
    """thermal_source='thermal': real thermal image. 'rgbgray': grayscale of the RGB image (control)."""
    def __init__(self, ann_path, rgb_dir, thermal_dir, thermal_source="thermal"):
        self.rgb_dir, self.thermal_dir, self.thermal_source = Path(rgb_dir), Path(thermal_dir), thermal_source
        data = json.loads(Path(ann_path).read_text())
        self.images_by_id = {img["id"]: img for img in data["images"]}
        self.image_ids = sorted(self.images_by_id.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        info = self.images_by_id[img_id]
        bgr = cv2.resize(cv2.imread(str(self.rgb_dir / info["file_name"])), (IMG_SIZE, IMG_SIZE))
        rgb = ((bgr.astype(np.float32) - NORM_MEAN) / NORM_STD).transpose(2, 0, 1)
        if self.thermal_source == "rgbgray":
            thermal = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        else:
            thermal = cv2.resize(cv2.imread(str(self.thermal_dir / info["file_name"]), cv2.IMREAD_GRAYSCALE),
                                 (IMG_SIZE, IMG_SIZE))
        thermal = (thermal.astype(np.float32) / 255.0)[np.newaxis]
        return {"rgb": rgb, "thermal": thermal, "img_id": img_id, "orig_w": info["width"], "orig_h": info["height"]}


def eval_collate_fn(batch):
    gt_meta = {
        "img": torch.stack([torch.from_numpy(it["rgb"]) for it in batch]),
        "gt_bboxes_ignore": None,
        "img_info": {"height": torch.tensor([it["orig_h"] for it in batch]),
                     "width": torch.tensor([it["orig_w"] for it in batch]),
                     "id": torch.tensor([it["img_id"] for it in batch])},
        "warp_matrix": [np.array([[IMG_SIZE / it["orig_w"], 0, 0], [0, IMG_SIZE / it["orig_h"], 0], [0, 0, 1]],
                                 dtype=np.float64) for it in batch],
    }
    return gt_meta, torch.stack([torch.from_numpy(it["thermal"]) for it in batch])


class ThermalBranch(nn.Module):
    """n_extra=0: the stage-0 branch (B and C). n_extra=1: the stage-1 branch used only in C."""
    def __init__(self, out_channels, n_extra=0, mid=16):
        super().__init__()
        chans = [mid, mid * 2] + [mid * 2] * n_extra + [out_channels]
        layers, c_in = [], 1
        for c_out in chans:
            layers += [nn.Conv2d(c_in, c_out, 3, 2, 1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]
            c_in = c_out
        self.net = nn.Sequential(*layers)
    def forward(self, thermal):
        return self.net(thermal)


class ChannelAttentionFusion(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(nn.Linear(channels * 2, channels // reduction), nn.ReLU(inplace=True),
                                  nn.Linear(channels // reduction, channels * 2), nn.Sigmoid())
    def forward(self, rgb_feat, thermal_feat):
        b, c, h, w = rgb_feat.shape
        wts = self.gate(torch.cat([self.pool(rgb_feat).view(b, c), self.pool(thermal_feat).view(b, c)], dim=1))
        return rgb_feat * wts[:, :c].view(b, c, 1, 1) + thermal_feat * wts[:, c:].view(b, c, 1, 1)


def build_frozen_base(model_cfg, config_a):
    load_config(cfg, str(model_cfg))
    base = build_model(cfg.model)
    ck = torch.load(config_a, map_location="cpu")
    base.load_state_dict({k.replace("avg_model.", "", 1): v for k, v in ck["state_dict"].items()
                          if k.startswith("avg_model.")}, strict=True)
    base.eval()
    for p in base.parameters():
        p.requires_grad = False
    return base


def attach_fusion(base, kind, ckpt_path, current):
    """Loads the trainable parts of a B or C checkpoint and hooks them onto the frozen backbone."""
    ck = torch.load(ckpt_path, map_location="cpu")
    if kind == "B":
        tb0, fm0 = ThermalBranch(116), ChannelAttentionFusion(116)
        tb0.load_state_dict(ck["thermal_branch"]); fm0.load_state_dict(ck["fusion_module"])
        mods = [tb0, fm0]
        def hook(module, inputs, output):
            f0, f1, f2 = output
            return (fm0(f0, tb0(current["x"])), f1, f2)
    else:
        tb0, fm0 = ThermalBranch(116), ChannelAttentionFusion(116)
        tb1, fm1 = ThermalBranch(232, n_extra=1), ChannelAttentionFusion(232)
        tb0.load_state_dict(ck["thermal_branch_0"]); fm0.load_state_dict(ck["fusion_module_0"])
        tb1.load_state_dict(ck["thermal_branch_1"]); fm1.load_state_dict(ck["fusion_module_1"])
        mods = [tb0, fm0, tb1, fm1]
        def hook(module, inputs, output):
            f0, f1, f2 = output
            return (fm0(f0, tb0(current["x"])), fm1(f1, tb1(current["x"])), f2)
    for m in mods:
        m.to(device).eval()
    base.backbone.register_forward_hook(hook)


def evaluate(setup, split, kind, ckpt_path, thermal_source):
    s = SETUPS[setup]
    ann = s["splits"][split]
    ds = PairedEvalDataset(ann, s["rgb_dir"], s["thermal_dir"], thermal_source)
    ds.coco_api = COCO(str(ann)); ds.class_names = s["class_names"]; ds.cat_ids = sorted(ds.coco_api.getCatIds())
    loader = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    base = build_frozen_base(s["model_cfg"], s["config_a"]).to(device)
    current = {}
    if kind != "A":
        attach_fusion(base, kind, ckpt_path, current)

    all_results = {}
    with torch.no_grad():
        for gt_meta, thermal_batch in loader:
            gt_meta["img"] = gt_meta["img"].to(device)
            current["x"] = thermal_batch.to(device)
            all_results.update(base.head.post_process(base(gt_meta["img"]), gt_meta))
    pyc = CocoDetectionEvaluator(ds).evaluate(all_results, save_dir=str(EVAL_SCRATCH))
    ours = nanodet_results_to_pr(all_results, ds.coco_api, ds.cat_ids, ds.class_names)
    del base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"pyc_map50": float(pyc["AP_50"]), "pyc_map50_95": float(pyc["mAP"]), **ours}

In [3]:
# Cell 3 -- run all 16 evaluations (~5-10 min); writes journal\results\nanodet_eval_all.json
results = {}
for setup, s in SETUPS.items():
    jobs = [(("A", None), ("A", None, "thermal"))] + list(s["models"].items())
    for split in s["splits"]:
        for (cfg_name, seed), (kind, ckpt_path, source) in jobs:
            print(f"\n=== {setup} {split} | {cfg_name} seed {seed} ===")
            r = evaluate(setup, split, kind, ckpt_path, source)
            results[f"{setup}|{split}|{cfg_name}|{seed}"] = r
            print(f"pycocotools mAP50={r['pyc_map50']:.4f}  mAP50:95={r['pyc_map50_95']:.4f}   "
                  f"P={r['precision']:.4f}  R={r['recall']:.4f}")

out = OUT_DIR / "nanodet_eval_all.json"
out.write_text(json.dumps(results, indent=2))
print(f"\nsaved: {out}")

print(f"\n{'setup':<6}{'split':<7}{'config':<11}{'seed':>5}{'mAP50':>9}{'mAP50:95':>10}{'P':>8}{'R':>8}")
for key, r in results.items():
    setup, split, cfg_name, seed = key.split("|")
    print(f"{setup:<6}{split:<7}{cfg_name:<11}{seed:>5}{r['pyc_map50']:>9.4f}{r['pyc_map50_95']:>10.4f}"
          f"{r['precision']:>8.4f}{r['recall']:>8.4f}")


=== flir night | A seed None ===
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.


C:\Users\LENOVO\miniconda3\envs\cv_gpu\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4384.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading and preparing results...
DONE (t=0.27s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.34s).
Accumulating evaluation results...
DONE (t=0.52s).
pycocotools mAP50=0.4081  mAP50:95=0.1660   P=0.5804  R=0.3634

=== flir night | B seed 1 ===
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
Loading and preparing results...
DONE (t=0.67s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.70s).
Accumulating evaluation results...
DONE (t=0.53s).
pycocotools mAP50=0.4928  mAP50:95=0.2162   P=0.5860  R=0.4554

=== flir night | B seed 2 ===
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
Loading and preparing results...
DONE (t=0.67s)
creating index...
index crea